In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
import random
import math

try:
    from IPython.display import display, clear_output
except ImportError:
    pass

CONFIG = {
    "VOLUME_PATH": "sliepner.npy",
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "BATCH_SIZE": 4,            
    "EPOCHS": 100,              
    "LR": 1e-3,
    "SCALE": 2,
    "DIFFUSION_STEPS": 100,     
    "UNET_DIM": 32,             
    "SAVE_DIR": "checkpoints_diffusion",
    "PLOT_TRANSPOSE": True,
    "SEED": 42,
    "RESUME_TRAINING": True,    
    "CLIP_PERCENTILE": 1.0,
    "FFT_LOSS_WEIGHT": 0.1,
    "GDL_LOSS_WEIGHT": 1.0,     # High weight for edges/faults
    "SCHEDULE": "cosine",
    "TRAIN_SPLIT_STRIDE": 2,
    "BLUR_SIGMA": 2.0           # Crucial: The blur kernel size
}

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
torch.manual_seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
random.seed(CONFIG["SEED"])

def to_np(x):
    return x.detach().cpu().numpy()

def transpose_and_plot(ax, img, title):
    if isinstance(img, torch.Tensor):
        arr = img.squeeze().detach().cpu().numpy()
    else:
        arr = np.array(img)
    if CONFIG["PLOT_TRANSPOSE"]:
        arr = arr.T
    
    robust_max = np.percentile(np.abs(arr), 100 - CONFIG["CLIP_PERCENTILE"])
    ax.imshow(arr, cmap="RdBu", aspect="auto", vmin=-robust_max, vmax=robust_max)
    ax.set_title(title)
    ax.axis("off")

def plot_loss_curves(history):
    epochs = range(1, len(history['mse']) + 1)
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.plot(epochs, history['mse'], label='Pixel Loss')
    plt.title('Reconstruction Loss')
    plt.grid(True, alpha=0.3)
    plt.subplot(1, 3, 2)
    plt.plot(epochs, history['fft'], label='FFT Loss', color='orange')
    plt.title('Spectral Loss')
    plt.grid(True, alpha=0.3)
    plt.subplot(1, 3, 3)
    plt.plot(epochs, history['gdl'], label='GDL Loss', color='green')
    plt.title('Gradient Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def get_blur_kernel(sigma, device):
    k_size = int(math.ceil(2 * sigma)) * 2 + 1
    k_half = (k_size - 1) * 0.5
    x = torch.linspace(-k_half, k_half, steps=k_size).to(device)
    pdf = torch.exp(-0.5 * (x / sigma).pow(2))
    kernel1d = pdf / pdf.sum()
    kernel2d = kernel1d.unsqueeze(1) @ kernel1d.unsqueeze(0)
    return kernel2d.unsqueeze(0).unsqueeze(0)

def apply_degradation(img, kernel, scale):
    pad = kernel.shape[-1] // 2
    blurred = F.conv2d(img, kernel, padding=pad)
    lr = F.interpolate(blurred, scale_factor=1/scale, mode='bicubic', align_corners=False)
    return lr

def compute_freq_loss(pred, target):
    fft_pred = torch.fft.fftn(pred, dim=(-2, -1))
    fft_target = torch.fft.fftn(target, dim=(-2, -1))
    log_amp_pred = torch.log(torch.abs(fft_pred) + 1e-8)
    log_amp_target = torch.log(torch.abs(fft_target) + 1e-8)
    return F.l1_loss(log_amp_pred, log_amp_target)

def compute_gradient_loss(pred, target):
    pred_dx = torch.abs(pred[:, :, :, :-1] - pred[:, :, :, 1:])
    target_dx = torch.abs(target[:, :, :, :-1] - target[:, :, :, 1:])
    pred_dy = torch.abs(pred[:, :, :-1, :] - pred[:, :, 1:, :])
    target_dy = torch.abs(target[:, :, :-1, :] - target[:, :, 1:, :])
    return F.l1_loss(pred_dx, target_dx) + F.l1_loss(pred_dy, target_dy)

class SeismicSRDataset(Dataset):
    def __init__(self, volume_path, scale=CONFIG["SCALE"], split='train'):
        if not os.path.exists(volume_path):
            vol = np.random.randn(100, 128, 128).astype(np.float32)
        else:
            vol = np.load(volume_path).astype(np.float32)
        
        # Robust Normalization
        self.norm_scale = np.percentile(np.abs(vol), 99.5)
        if self.norm_scale < 1e-6: self.norm_scale = 1.0 
        print(f"Dataset {split}: Norm Scale {self.norm_scale:.4f}")
        
        vol = vol / self.norm_scale
        vol = np.clip(vol, -1.0, 1.0)
        
        stride = CONFIG["TRAIN_SPLIT_STRIDE"]
        if split == 'train':
            self.slices = [vol[i] for i in range(0, vol.shape[0], stride)]
        else:
            self.slices = [vol[i] for i in range(vol.shape[0]) if i % stride != 0]
            
        self.scale = scale
        # Pre-compute blur kernel
        self.blur_kernel = get_blur_kernel(CONFIG["BLUR_SIGMA"], "cpu")

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        hr = self.slices[idx]
        hr = torch.tensor(hr, dtype=torch.float32).unsqueeze(0)
        hr = hr.unsqueeze(0) # [1, 1, H, W]
        
        # Apply Realistic Degradation (Blur + Downsample)
        lr_small = apply_degradation(hr, self.blur_kernel, self.scale)
        
        # Upsample for conditioning
        lr_up = F.interpolate(lr_small, size=hr.shape[2:], mode='bicubic', align_corners=False)
        
        return lr_up.squeeze(0), hr.squeeze(0), lr_small.squeeze(0)

# ---------------------------------------------------------
# Diffusion & U-Net Components
# ---------------------------------------------------------

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class Block(nn.Module):
    def __init__(self, dim, dim_out, groups=8):
        super().__init__()
        self.proj = nn.Conv2d(dim, dim_out, 3, padding=1)
        self.norm = nn.GroupNorm(groups, dim_out)
        self.act = nn.SiLU()
    def forward(self, x, scale_shift=None):
        x = self.proj(x)
        x = self.norm(x)
        if scale_shift is not None:
            scale, shift = scale_shift
            x = x * (scale + 1) + shift
        return self.act(x)

class ResnetBlock(nn.Module):
    def __init__(self, dim, dim_out, *, time_emb_dim=None, groups=8):
        super().__init__()
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, dim_out * 2)) if time_emb_dim else None
        self.block1 = Block(dim, dim_out, groups=groups)
        self.block2 = Block(dim_out, dim_out, groups=groups)
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()
    def forward(self, x, time_emb=None):
        scale_shift = None
        if self.mlp is not None and time_emb is not None:
            time_emb = self.mlp(time_emb)
            time_emb = time_emb.unsqueeze(-1).unsqueeze(-1)
            scale_shift = time_emb.chunk(2, dim=1)
        h = self.block1(x, scale_shift=scale_shift)
        h = self.block2(h)
        return h + self.res_conv(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        hidden_dim = dim_head * heads
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias=False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)
    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=1)
        q, k, v = map(lambda t: t.reshape(b, self.heads, -1, h * w), qkv)
        q = q * self.scale
        sim = torch.einsum('b h d i, b h d j -> b h i j', q, k)
        attn = sim.softmax(dim=-1)
        out = torch.einsum('b h i j, b h d j -> b h d i', attn, v)
        out = out.reshape(b, -1, h, w)
        return self.to_out(out) + x

class TimestepEmbedSequential(nn.Sequential):
    def forward(self, x, emb):
        for layer in self:
            if isinstance(layer, ResnetBlock): x = layer(x, emb)
            else: x = layer(x)
        return x

class UNet(nn.Module):
    def __init__(self, dim=CONFIG["UNET_DIM"], out_dim=1, channels=2): 
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(dim), nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim * 4),
        )
        self.init_conv = nn.Conv2d(channels, dim, 7, padding=3)
        self.downs = nn.ModuleList([
            TimestepEmbedSequential(ResnetBlock(dim, dim, time_emb_dim=dim*4), ResnetBlock(dim, dim, time_emb_dim=dim*4)),
            TimestepEmbedSequential(ResnetBlock(dim, dim*2, time_emb_dim=dim*4), ResnetBlock(dim*2, dim*2, time_emb_dim=dim*4)),
            TimestepEmbedSequential(ResnetBlock(dim*2, dim*4, time_emb_dim=dim*4), ResnetBlock(dim*4, dim*4, time_emb_dim=dim*4)),
        ])
        self.down_samples = nn.ModuleList([nn.Conv2d(dim, dim, 4, 2, 1), nn.Conv2d(dim*2, dim*2, 4, 2, 1), nn.Conv2d(dim*4, dim*4, 4, 2, 1)])
        self.mid_block1 = ResnetBlock(dim*4, dim*4, time_emb_dim=dim*4)
        self.mid_attn = Attention(dim*4) 
        self.mid_block2 = ResnetBlock(dim*4, dim*4, time_emb_dim=dim*4)
        self.ups = nn.ModuleList([
            TimestepEmbedSequential(ResnetBlock(dim*8, dim*2, time_emb_dim=dim*4), ResnetBlock(dim*2, dim*2, time_emb_dim=dim*4)),
            TimestepEmbedSequential(ResnetBlock(dim*4, dim, time_emb_dim=dim*4), ResnetBlock(dim, dim, time_emb_dim=dim*4)),
            TimestepEmbedSequential(ResnetBlock(dim*2, dim, time_emb_dim=dim*4), ResnetBlock(dim, dim, time_emb_dim=dim*4)),
        ])
        self.up_samples = nn.ModuleList([nn.ConvTranspose2d(dim*4, dim*4, 4, 2, 1), nn.ConvTranspose2d(dim*2, dim*2, 4, 2, 1), nn.ConvTranspose2d(dim, dim, 4, 2, 1)])
        self.final_res_block = ResnetBlock(dim, dim, time_emb_dim=dim*4)
        self.final_conv = nn.Conv2d(dim, out_dim, 1)

    def forward(self, x, time, cond):
        x = torch.cat([x, cond], dim=1)
        t = self.time_mlp(time)
        x = self.init_conv(x)
        h_vals = []
        for block, sample in zip(self.downs, self.down_samples):
            x = block(x, t); h_vals.append(x); x = sample(x)
        x = self.mid_block1(x, t); x = self.mid_attn(x); x = self.mid_block2(x, t)
        for block, sample in zip(self.ups, self.up_samples):
            x = sample(x); h = h_vals.pop()
            if x.shape[2:] != h.shape[2:]: x = F.interpolate(x, size=h.shape[2:], mode='nearest')
            x = torch.cat((x, h), dim=1); x = block(x, t)
        return self.final_conv(self.final_res_block(x, t))

class Diffusion(nn.Module):
    def __init__(self, model, steps=CONFIG["DIFFUSION_STEPS"], device=CONFIG["DEVICE"]):
        super().__init__()
        self.model = model
        self.steps = steps
        self.device = device
        
        # Cosine Schedule
        s=0.008
        x = torch.linspace(0, steps, steps + 1)
        alphas_cumprod = torch.cos(((x / steps) + s) / (1 + s) * math.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        self.betas = torch.clip(1 - (alphas_cumprod[1:] / alphas_cumprod[:-1]), 0.0001, 0.9999).to(device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0).to(device)
        
        # Helper vars
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.posterior_mean_coef1 = (self.betas * torch.sqrt(alphas_cumprod_prev) / (1. - self.alphas_cumprod))
        self.posterior_mean_coef2 = ((1. - alphas_cumprod_prev) * torch.sqrt(self.alphas) / (1. - self.alphas_cumprod))
        self.posterior_variance = (self.betas * (1. - alphas_cumprod_prev) / (1. - self.alphas_cumprod))

    def q_sample(self, x_0, t, noise=None):
        if noise is None: noise = torch.randn_like(x_0)
        return (self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1) * x_0 + 
                self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) * noise), noise

    @torch.no_grad()
    def p_sample(self, x, t, t_index, cond):
        pred_x0 = torch.clamp(self.model(x, t, cond), -1., 1.)
        mean = (self.posterior_mean_coef1[t].view(-1, 1, 1, 1) * pred_x0 + 
                self.posterior_mean_coef2[t].view(-1, 1, 1, 1) * x)
        if t_index == 0: return mean
        noise = torch.randn_like(x)
        return mean + torch.sqrt(self.posterior_variance[t].view(-1, 1, 1, 1)) * noise

    @torch.no_grad()
    def sample(self, cond):
        b, c, h, w = cond.shape
        img = torch.randn((b, 1, h, w), device=self.device)
        for i in tqdm(reversed(range(0, self.steps)), desc='Sampling', total=self.steps, leave=False):
            t = torch.full((b,), i, device=self.device, dtype=torch.long)
            img = self.p_sample(img, t, i, cond)
        return img

def compute_metrics(sr, hr):
    sr_np = sr.detach().cpu().squeeze().numpy(); hr_np = hr.detach().cpu().squeeze().numpy()
    if sr_np.ndim == 2:
        dr = hr_np.max() - hr_np.min()
        return (psnr_metric(hr_np, sr_np, data_range=dr), ssim_metric(hr_np, sr_np, data_range=dr), np.mean(np.abs(sr_np - hr_np)))
    else:
        p, s, m = 0, 0, 0
        for i in range(sr_np.shape[0]):
            dr = hr_np[i].max() - hr_np[i].min()
            p += psnr_metric(hr_np[i], sr_np[i], data_range=dr)
            s += ssim_metric(hr_np[i], sr_np[i], data_range=dr)
            m += np.mean(np.abs(sr_np[i] - hr_np[i]))
        return p/sr_np.shape[0], s/sr_np.shape[0], m/sr_np.shape[0]

def train():
    train_ds = SeismicSRDataset(CONFIG["VOLUME_PATH"], scale=CONFIG["SCALE"], split='train')
    val_ds = SeismicSRDataset(CONFIG["VOLUME_PATH"], scale=CONFIG["SCALE"], split='val')
    loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, drop_last=True)
    
    unet = UNet().to(CONFIG["DEVICE"])
    diffusion = Diffusion(unet).to(CONFIG["DEVICE"])
    opt = torch.optim.Adam(unet.parameters(), lr=CONFIG["LR"])

    start_epoch = 1
    best_psnr = -1.0
    history = {'mse': [], 'fft': [], 'gdl': []}
    
    if CONFIG["RESUME_TRAINING"] and os.path.exists(os.path.join(CONFIG["SAVE_DIR"], "diffusion_best.pth")):
        try:
            ckpt = torch.load(os.path.join(CONFIG["SAVE_DIR"], "diffusion_best.pth"), map_location=CONFIG["DEVICE"], weights_only=False)
            unet.load_state_dict(ckpt['model'])
            opt.load_state_dict(ckpt['opt'])
            start_epoch = ckpt['epoch'] + 1
            best_psnr = ckpt.get('best_psnr', -1.0)
            print(f"Resumed from Epoch {start_epoch-1}. Best PSNR: {best_psnr:.4f}")
        except: pass

    val_indices = sorted(list(set([min(5, len(val_ds)-1), len(val_ds)//2, len(val_ds)-1]))) if len(val_ds) > 0 else []

    for epoch in range(start_epoch, CONFIG["EPOCHS"]+1):
        unet.train()
        avg_loss = {'mse': 0., 'fft': 0., 'gdl': 0.}
        pbar = tqdm(loader, desc=f"Ep {epoch}")
        
        for lr_up, hr, _ in pbar:
            lr_up, hr = lr_up.to(CONFIG["DEVICE"]), hr.to(CONFIG["DEVICE"])
            t = torch.randint(0, CONFIG["DIFFUSION_STEPS"], (hr.shape[0],), device=CONFIG["DEVICE"]).long()
            noisy_hr, _ = diffusion.q_sample(hr, t)
            
            if noisy_hr.shape != lr_up.shape:
                lr_up = F.interpolate(lr_up, size=noisy_hr.shape[2:], mode='bilinear')

            pred_x0 = unet(noisy_hr, t, lr_up)
            
            losses = {
                'mse': F.mse_loss(pred_x0, hr),
                'fft': compute_freq_loss(pred_x0, hr),
                'gdl': compute_gradient_loss(pred_x0, hr)
            }
            total_loss = losses['mse'] + CONFIG["FFT_LOSS_WEIGHT"]*losses['fft'] + CONFIG["GDL_LOSS_WEIGHT"]*losses['gdl']
            
            opt.zero_grad(); total_loss.backward(); opt.step()
            
            for k in avg_loss: avg_loss[k] += losses[k].item()
            pbar.set_postfix({k: f"{v.item():.4f}" for k,v in losses.items()})

        for k in avg_loss: history[k].append(avg_loss[k] / len(loader))
        
        unet.eval()
        with torch.no_grad():
            try: clear_output(wait=True)
            except: pass
            plot_loss_curves(history)
            
            epoch_psnr = 0
            for v_idx in val_indices:
                lr_up, hr, _ = val_ds[v_idx]
                lr_up, hr = lr_up.unsqueeze(0).to(CONFIG["DEVICE"]), hr.unsqueeze(0).to(CONFIG["DEVICE"])
                sr = diffusion.sample(lr_up)
                if sr.shape != hr.shape: sr = F.interpolate(sr, size=hr.shape[2:], mode='bilinear')
                
                # Denorm
                s_d, h_d, l_d = sr * val_ds.norm_scale, hr * val_ds.norm_scale, lr_up * val_ds.norm_scale
                p_sr, _, m_sr = compute_metrics(s_d, h_d)
                p_base, _, m_base = compute_metrics(l_d, h_d)
                epoch_psnr += p_sr
                
                try:
                    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
                    transpose_and_plot(ax[0], l_d[0,0], f"Baseline\nPSNR: {p_base:.2f}")
                    transpose_and_plot(ax[1], h_d[0,0], "Target")
                    transpose_and_plot(ax[2], s_d[0,0], f"Diffusion\nPSNR: {p_sr:.2f}")
                    plt.savefig(os.path.join(CONFIG["SAVE_DIR"], f"ep{epoch}_idx{v_idx}.png"))
                    plt.show()
                except: pass
            
            avg_p = epoch_psnr / len(val_indices)
            print(f"Epoch {epoch} Val PSNR: {avg_p:.4f}")
            
            if avg_p > best_psnr:
                best_psnr = avg_p
                torch.save({'epoch': epoch, 'model': unet.state_dict(), 'opt': opt.state_dict(), 'best_psnr': best_psnr}, 
                           os.path.join(CONFIG["SAVE_DIR"], "diffusion_best.pth"))

if __name__ == "__main__":
    train()